# Step 2.2 - Structured Data EDA

This notebook covers **Step 2.2 - Structured Data EDA**.

Current local snapshot scope:
- full structured EDA for the 3 development projects: `department-of-premier-and-cabinet-wa`, `suncorp-insurance`, `the-university-of-queensland`
- survey-only coverage for `bupa-uk`
- no local Brighton metadata bundle in the current snapshot

Goals:
1. Audit raw-data coverage across the intended Step 2.2 scope
2. Preserve the main structured EDA for the 3 development projects
3. Add survey EDA for locally available survey bundles
4. Document current source-data limitations explicitly


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

RAW_DIR = BASE_DIR / "data" / "raw"
PROC_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"


In [ ]:
# Load processed and raw structured data
windows = pd.read_csv(PROC_DIR / "windows.csv")
video_metadata = pd.read_csv(PROC_DIR / "video_metadata.csv")
tester_db = pd.read_csv(RAW_DIR / "tester_db.csv")

# raw tasks for the 3 development projects
tasks_dep = pd.read_csv(RAW_DIR / "department-of-premier-and-cabinet-wa" / "coercive-control-support-tasks.csv")
tasks_sun = pd.read_csv(RAW_DIR / "suncorp-insurance" / "aami-website-tasks.csv")
tasks_uq  = pd.read_csv(RAW_DIR / "the-university-of-queensland" / "postgrad-enrolment-experience-tasks.csv")

tasks_dep["project"] = "department-of-premier-and-cabinet-wa"
tasks_sun["project"] = "suncorp-insurance"
tasks_uq["project"] = "the-university-of-queensland"

tasks_df = pd.concat([tasks_dep, tasks_sun, tasks_uq], ignore_index=True)

DEV_PROJECTS = [
    "department-of-premier-and-cabinet-wa",
    "suncorp-insurance",
    "the-university-of-queensland",
]

PROJECT_SPECS = [
    {
        "project": "department-of-premier-and-cabinet-wa",
        "folder": "department-of-premier-and-cabinet-wa",
        "projects_csv": "department-of-premier-and-cabinet-wa-projects.csv",
        "tasks_csv": "coercive-control-support-tasks.csv",
        "assignments_csv": "coercive-control-support-assignments.csv",
        "survey_prefix": None,
    },
    {
        "project": "suncorp-insurance",
        "folder": "suncorp-insurance",
        "projects_csv": "suncorp-insurance-projects.csv",
        "tasks_csv": "aami-website-tasks.csv",
        "assignments_csv": "aami-website-assignments.csv",
        "survey_prefix": "aami-website",
    },
    {
        "project": "the-university-of-queensland",
        "folder": "the-university-of-queensland",
        "projects_csv": "the-university-of-queensland-projects.csv",
        "tasks_csv": "postgrad-enrolment-experience-tasks.csv",
        "assignments_csv": "postgrad-enrolment-experience-assignments.csv",
        "survey_prefix": "postgrad-enrolment-experience",
    },
    {
        "project": "bupa-uk",
        "folder": "bupa-uk",
        "projects_csv": None,
        "tasks_csv": None,
        "assignments_csv": None,
        "survey_prefix": "web-health-information",
    },
    {
        "project": "brighton",
        "folder": "brighton",
        "projects_csv": None,
        "tasks_csv": None,
        "assignments_csv": None,
        "survey_prefix": None,
    },
]

print("windows columns:", windows.columns.tolist())
print("video_metadata columns:", video_metadata.columns.tolist())
print("tasks_df columns:", tasks_df.columns.tolist())
print("tester_db columns:", tester_db.columns.tolist())


In [ ]:
# Basic cleaning and type normalization
windows["tester_name"] = windows["tester_name"].astype(str).str.strip()
windows["project"] = windows["project"].astype(str).str.strip()

video_metadata["project"] = video_metadata["project"].astype(str).str.strip()
video_metadata["tester_name"] = video_metadata["tester_name"].astype(str).str.strip()
video_metadata["duration_ratio"] = pd.to_numeric(video_metadata["duration_ratio"], errors="coerce")
video_metadata["duration_seconds"] = pd.to_numeric(video_metadata["duration_seconds"], errors="coerce")

tasks_df["project"] = tasks_df["project"].astype(str).str.strip()
tasks_df["Order"] = pd.to_numeric(tasks_df["Order"], errors="coerce")
tasks_df["TaskType"] = tasks_df["TaskType"].astype(str).str.strip()
tasks_df["Timeguide"] = tasks_df["Timeguide"].astype(str).str.strip()

tester_db["Name"] = tester_db["Name"].astype(str).str.strip()


In [ ]:
# Helper: convert Timeguide text to numeric minutes
def parse_timeguide_to_minutes(x):
    if pd.isna(x):
        return None
    
    x = str(x).strip().lower()
    if x == "" or x == "nan":
        return None
    
    # examples:
    # "10 mins max"
    # "15 mins max "
    # "30 seconds or less"
    # "1 minute"
    num_match = re.search(r"(\d+(\.\d+)?)", x)
    if not num_match:
        return None
    
    value = float(num_match.group(1))
    
    if "second" in x:
        return value / 60
    if "min" in x:
        return value
    if "hour" in x:
        return value * 60
    
    return value

tasks_df["Timeguide_minutes"] = tasks_df["Timeguide"].apply(parse_timeguide_to_minutes)

## Coverage Check

Before the main EDA, audit the current local raw snapshot across the intended Step 2.2 scope.
This separates **full structured EDA**, **survey-only analysis**, and **not currently available locally**.


In [ ]:
coverage_rows = []

for spec in PROJECT_SPECS:
    folder = RAW_DIR / spec["folder"]
    survey_prefix = spec["survey_prefix"]
    survey_files = []
    if survey_prefix:
        survey_files = [
            f"{survey_prefix}-surveys.csv",
            f"{survey_prefix}-survey-questions.csv",
            f"{survey_prefix}-survey-responses.csv",
            f"{survey_prefix}-survey-answers.csv",
        ]

    has_folder = folder.exists()
    has_projects_csv = bool(spec["projects_csv"]) and (folder / spec["projects_csv"]).exists()
    has_tasks_csv = bool(spec["tasks_csv"]) and (folder / spec["tasks_csv"]).exists()
    has_assignments_csv = bool(spec["assignments_csv"]) and (folder / spec["assignments_csv"]).exists()
    survey_files_present = sum((folder / name).exists() for name in survey_files)
    has_full_survey_bundle = bool(survey_files) and survey_files_present == len(survey_files)

    if spec["project"] in DEV_PROJECTS and has_projects_csv and has_tasks_csv and has_assignments_csv:
        analysis_scope = "full structured EDA"
    elif has_full_survey_bundle:
        analysis_scope = "survey EDA only"
    else:
        analysis_scope = "coverage check only"

    if spec["project"] == "suncorp-insurance" and not has_full_survey_bundle:
        note = "AAMI survey bundle not present in current local snapshot"
    elif spec["project"] == "bupa-uk":
        note = "Survey bundle only; no local projects/tasks/assignments files"
    elif spec["project"] == "brighton":
        note = "No local Brighton folder in the current snapshot"
    elif has_assignments_csv and spec["project"] in DEV_PROJECTS:
        note = "Assignments export is tester-roster style; task-level linkage unavailable"
    else:
        note = ""

    coverage_rows.append({
        "project": spec["project"],
        "has_folder": has_folder,
        "has_projects_csv": has_projects_csv,
        "has_tasks_csv": has_tasks_csv,
        "has_assignments_csv": has_assignments_csv,
        "survey_files_present": survey_files_present,
        "has_full_survey_bundle": has_full_survey_bundle,
        "analysis_scope": analysis_scope,
        "notes": note,
    })

coverage_df = pd.DataFrame(coverage_rows)
print("=== Local raw-data coverage by project ===")
display(coverage_df)


### Section 1 Takeaways - Coverage

- The current local snapshot supports **full structured EDA** only for the 3 development projects.
- `bupa-uk` currently contributes a **survey-only** bundle.
- Brighton is excluded from structured-data EDA because only raw videos are available locally.
- Tester-to-task linkage is not recoverable from the current `*-assignments.csv` exports.


## Development-Project Structured EDA

The main structured EDA below remains scoped to the 3 development projects because they are the only projects with local `projects/tasks/assignments` files in the current snapshot.


## 1. Project Scale Overview

Count testers and tasks across the 3 development projects before moving into cross-project patterns.


In [ ]:
# 1) Count testers and tasks across the 3 development projects

tester_per_project = (
    windows.groupby("project")["tester_name"]
    .nunique()
    .reset_index(name="tester_count")
)

task_per_project = (
    tasks_df.groupby("project")["Order"]
    .nunique()
    .reset_index(name="task_count")
)

project_overview = tester_per_project.merge(task_per_project, on="project", how="outer")

print("=== Number of testers and tasks by project ===")
display(project_overview)

### Section 2 Takeaways - Project Scale

- Tester counts and task counts establish the structural baseline for the development projects.
- These project-level differences should be treated as context when interpreting later task-design and duration findings.


## 2. Cross-Project Tester Participation

Check how many testers appear across multiple development projects and whether cross-project tracking is plausible.


In [ ]:
# 2) Analyze cross-project tester participation

tester_cross_project = (
    windows.groupby("tester_name")["project"]
    .nunique()
    .reset_index(name="project_count")
    .sort_values(["project_count", "tester_name"], ascending=[False, True])
)

cross_project_testers = tester_cross_project[tester_cross_project["project_count"] > 1]

print("=== Tester cross-project participation ===")
display(tester_cross_project)

print("=== Testers involved in more than one project ===")
display(cross_project_testers)

print("Number of cross-project testers:", len(cross_project_testers))

### Section 3 Takeaways - Cross-Project Participation

- Cross-project overlap is a meaningful structural property of the dev snapshot.
- It supports later comparison of tester behaviour across projects, but not tester-to-task linkage.


## 3. Task Design Patterns

Review task-type labels and Timeguide patterns for the 3 development projects.


In [ ]:
# 3) Review task-type distribution in the 3 development projects

task_type_dist = (
    tasks_df["TaskType"]
    .replace({"nan": pd.NA, "": pd.NA})
    .value_counts(dropna=False)
    .reset_index()
)
task_type_dist.columns = ["TaskType", "count"]

print("=== Task type distribution ===")
display(task_type_dist)

In [ ]:
plt.figure(figsize=(8, 5))
tasks_df["TaskType"].replace({"nan": pd.NA, "": pd.NA}).value_counts(dropna=False).plot(kind="bar")
plt.title("Task Type Distribution")
plt.xlabel("Task Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 3) Summarize Timeguide values for the 3 development projects

print("=== Timeguide summary (minutes) ===")
display(tasks_df["Timeguide_minutes"].describe())

display(tasks_df[["project", "Title", "Timeguide", "Timeguide_minutes"]].head(10))

### Section 4 Takeaways - Task Design

- Task labels are fragmented and may need normalisation before later modelling.
- Timeguide should be interpreted as loose context rather than as a precise expected-duration target.


In [ ]:
timeguide_counts = tasks_df["Timeguide_minutes"].value_counts().sort_index()

display(timeguide_counts)

plt.figure(figsize=(8, 5))
timeguide_counts.plot(kind="bar")
plt.title("Timeguide Distribution")
plt.xlabel("Timeguide (minutes)")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# optional: Timeguide by project
timeguide_by_project = (
    tasks_df.groupby("project")["Timeguide_minutes"]
    .describe()
)

print("=== Timeguide distribution by project ===")
display(timeguide_by_project)

## 4. Duration Deviation

Compare observed video duration against Timeguide to identify structural mismatch before later rule design.


In [ ]:
# 4) Compare actual video duration against Timeguide
# `duration_ratio` is already available in `video_metadata.csv`

print("=== Duration ratio summary ===")
display(video_metadata["duration_ratio"].describe())

### Section 5 Takeaways - Duration Deviation

- Duration-ratio analysis remains one of the strongest structural findings in Step 2.2.
- Large deviations support later anomaly-style validation, but do not imply that Timeguide is a strict quality baseline.


In [ ]:
# optional: identify extreme deviations, aligned with later Layer 1 threshold idea
duration_anomalies = video_metadata[
    (video_metadata["duration_ratio"] < 0.3) |
    (video_metadata["duration_ratio"] > 3.0)
].copy()

print("=== Videos with extreme duration deviation ===")
display(duration_anomalies[["project", "tester_name", "video_filename", "duration_seconds", "duration_ratio"]])

## 5. Survey EDA

Survey analysis is limited to bundles that are actually available in the current local snapshot.


In [ ]:
SURVEY_SPECS = [
    {"project": "suncorp-insurance", "label": "AAMI / Suncorp", "folder": "suncorp-insurance", "prefix": "aami-website"},
    {"project": "the-university-of-queensland", "label": "UQ", "folder": "the-university-of-queensland", "prefix": "postgrad-enrolment-experience"},
    {"project": "bupa-uk", "label": "Bupa", "folder": "bupa-uk", "prefix": "web-health-information"},
]

def load_survey_bundle(folder_name, prefix):
    folder = RAW_DIR / folder_name
    paths = {
        "surveys": folder / f"{prefix}-surveys.csv",
        "questions": folder / f"{prefix}-survey-questions.csv",
        "responses": folder / f"{prefix}-survey-responses.csv",
        "answers": folder / f"{prefix}-survey-answers.csv",
    }
    if not all(p.exists() for p in paths.values()):
        return None
    return {name: pd.read_csv(p) for name, p in paths.items()}

survey_project_rows = []
survey_question_type_rows = []
missing_survey_labels = []

for spec in SURVEY_SPECS:
    bundle = load_survey_bundle(spec["folder"], spec["prefix"])
    if bundle is None:
        missing_survey_labels.append(spec["label"])
        continue

    surveys = bundle["surveys"]
    questions = bundle["questions"]
    responses = bundle["responses"]
    answers = bundle["answers"]

    completed_mask = responses["status"].astype(str).str.lower().eq("completed") if "status" in responses.columns else pd.Series(dtype=bool)
    answers_per_response = answers.groupby("response_id").size() if not answers.empty else pd.Series(dtype="int64")
    mandatory_count = questions[questions["is_mandatory"].astype(str).str.lower().eq("true")].shape[0] if "is_mandatory" in questions.columns else pd.NA

    survey_project_rows.append({
        "project": spec["project"],
        "survey_label": spec["label"],
        "survey_count": len(surveys),
        "question_count": len(questions),
        "response_count": len(responses),
        "completed_response_count": int(completed_mask.sum()) if len(responses) else 0,
        "answer_count": len(answers),
        "mandatory_question_count": mandatory_count,
        "mean_answers_per_response": round(float(answers_per_response.mean()), 2) if not answers_per_response.empty else 0.0,
        "median_answers_per_response": round(float(answers_per_response.median()), 2) if not answers_per_response.empty else 0.0,
    })

    qtype = questions["question_type"].fillna("missing").value_counts().rename_axis("question_type").reset_index(name="count")
    qtype["project"] = spec["project"]
    qtype["survey_label"] = spec["label"]
    survey_question_type_rows.append(qtype)

survey_project_summary = pd.DataFrame(survey_project_rows)
survey_question_type_summary = pd.concat(survey_question_type_rows, ignore_index=True) if survey_question_type_rows else pd.DataFrame(columns=["question_type", "count", "project", "survey_label"])

print("=== Available survey bundles in the local snapshot ===")
display(survey_project_summary)

print("=== Survey question-type distribution ===")
display(survey_question_type_summary.sort_values(["survey_label", "count"], ascending=[True, False]))

print("Missing local survey bundles:", missing_survey_labels)


### Section 6 Takeaways - Survey Coverage

- UQ and Bupa can be compared at the survey-structure level in the current local snapshot.
- AAMI / Suncorp survey files are not currently present locally, so survey conclusions should be framed as partial coverage.


In [ ]:
# Save key Step 2.2 summary tables
OUTPUT_DIR.mkdir(exist_ok=True)

project_overview.to_csv(OUTPUT_DIR / "step2_2_project_overview.csv", index=False)
tester_cross_project.to_csv(OUTPUT_DIR / "step2_2_tester_cross_project.csv", index=False)
task_type_dist.to_csv(OUTPUT_DIR / "step2_2_task_type_distribution.csv", index=False)
tasks_df.to_csv(OUTPUT_DIR / "step2_2_tasks_with_timeguide_minutes.csv", index=False)
coverage_df.to_csv(OUTPUT_DIR / "step2_2_data_coverage.csv", index=False)
survey_project_summary.to_csv(OUTPUT_DIR / "step2_2_survey_project_summary.csv", index=False)
survey_question_type_summary.to_csv(OUTPUT_DIR / "step2_2_survey_question_types.csv", index=False)

print("Saved Step 2.2 outputs to:", OUTPUT_DIR)


## Current Limitations

- Brighton currently has raw videos only, so it cannot be included in structured-data EDA.
- Bupa is represented locally by survey files only; no local `projects/tasks/assignments` bundle is available in this snapshot.
- The development-project `*-assignments.csv` exports behave like tester rosters rather than tester-to-task assignment tables, so tester-to-task linkage is not recoverable from the current files.
- AAMI / Suncorp survey files are not present in the local snapshot, so survey comparison is limited to UQ and Bupa for now.


## Step 2.2 Summary

This notebook separates **coverage audit**, **development-project structured EDA**, and **survey EDA** based on the current local snapshot rather than assuming all planned files are present.

- Full structured EDA is currently supported for the 3 development projects: WA, Suncorp/AAMI, and UQ.
- Bupa currently supports **survey-only** analysis in the local snapshot.
- Brighton currently has **raw videos only** and is therefore excluded from structured-data EDA until supporting metadata is available.
- The `*-assignments.csv` exports for the development projects behave like tester rosters rather than tester-to-task assignment tables, so task-level tester linkage should not be assumed.

The original structured EDA findings for the 3 development projects remain the main quantitative backbone: tester counts are broadly comparable, task types are fragmented, UQ timeguides are generally longer, and actual recording duration substantially exceeds Timeguide expectations.

Survey EDA is now scoped to the bundles that are actually available locally. In the current snapshot, UQ and Bupa can be compared at the survey-structure level through counts of surveys, questions, responses, answers, and question-type mix, while AAMI / Suncorp survey files remain absent locally.
